# New York walkthrough — weekly Handle and GGR

Official source: [NY Gaming Commission revenue reports](https://gaming.ny.gov/revenue-reports).

New York publishes **weekly** mobile sports-wagering Excel files. We keep weeks as weeks.
We do **not** spread a week across calendar months.

| Term | Meaning |
| --- | --- |
| Handle | Amount wagered that week |
| GGR | Gross gaming revenue as reported by the Commission (stored in `gross_revenue`) |
| Hold | GGR / handle (analysis only) |

GGR is **cash basis** in New York. Futures can be taxed when written; winning tickets when redeemed.
That makes weekly GGR jump around. **Negative GGR is valid** and is preserved.

This notebook does **not** map operators to FanDuel / FLUT or produce a forecast.

## 1. Discover official workbook links

The landing page lists a STATEWIDE weekly Excel and one weekly Excel per operator.
Discovery uses the Sports Wagering table — no hard-coded dated `system/files` URLs.

`run_live_discovery = False` keeps this notebook offline by default. The next cell parses a retained workbook. Set the flag to `True` only for an intentional live index request. Database inspection uses the explicitly selected staging snapshot read-only.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import http_get, project_root
from variant_gaming.states.new_york import LANDING_URL, discover_ny_sports_workbook_links, parse_ny_workbook
from variant_gaming.storage import connect_readonly

ROOT = project_root()
database_file = "data/staging/gaming_nationwide.sqlite"  # explicitly select another snapshot if needed
database_path = ROOT / database_file
print("Read-only database:", database_path)
run_live_discovery = False  # opt in only when a fresh official index is wanted
if run_live_discovery:
    landing = http_get(LANDING_URL)
    discovery = discover_ny_sports_workbook_links(landing.text)
    print("Statewide:", discovery["statewide"]["discovered_url"])
    display(pd.DataFrame(discovery["operators"])[["source_operator_name"]].head(12))
else:
    print("Live discovery disabled. The retained workbook below runs offline.")


## 2. Parse a saved official sample

The production collector downloads each workbook, hashes it, and upserts weekly rows.
Here we parse the test fixture so the transformation stays visible without dumping thousands of rows.

In [ ]:
fixture = ROOT / "tests" / "fixtures" / "NY" / "sample_statewide_weekly.xlsx"
parsed, sheet_check = parse_ny_workbook(fixture.read_bytes())
print("parsed rows", len(parsed))
print("sheet reconciliation rows", len(sheet_check))
print(parsed.columns.tolist())
parsed.head()

## 3. Validation ideas (already implemented in the module)

- Completed weeks have handle and GGR
- Handle is positive; GGR may be negative
- Sheet weekly cells reconcile to the published Total
- Operator vs statewide comparison is labeled complete / incomplete / mismatch — never forced to balance
- Frequency stays `weekly`; `row_type` is `official_statewide_total` or `operator`

In [ ]:
ny = pd.DataFrame()
if database_path.exists():
    conn = connect_readonly(database_path)
    try:
        ny = pd.read_sql_query("SELECT row_type, COUNT(*) AS n, MIN(period_start) AS earliest, MAX(period_end) AS latest FROM gaming_results WHERE state_code='NY' GROUP BY row_type", conn)
    finally:
        conn.close()
else:
    print("Selected database does not exist; inspection skipped. No database was created.")
ny


Full collection is run later in `20_run_all_collectors.ipynb`, which upserts and does not replace other states.